In [25]:
import sys
import os
# only go up a folder if we are currently inside the notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    
project_root = os.getcwd() 
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Phase 1: Verification
Verification if all files are imported successfully.

In [27]:
import pandas as pd

In [28]:
reviews_df = pd.read_csv("csv/tokopedia_product_reviews_2025.csv")
reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65543 entries, 0 to 65542
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   review_text       65543 non-null  str  
 1   review_date       65543 non-null  str  
 2   review_id         65543 non-null  int64
 3   product_name      65543 non-null  str  
 4   product_category  65543 non-null  str  
 5   product_variant   26749 non-null  str  
 6   product_price     65543 non-null  int64
 7   product_url       65543 non-null  str  
 8   product_id        65543 non-null  int64
 9   rating            65543 non-null  int64
 10  sold_count        65543 non-null  int64
 11  shop_id           65543 non-null  int64
 12  sentiment_label   65543 non-null  str  
dtypes: int64(6), str(7)
memory usage: 6.5 MB


65543 rows and 13 columns, column names seems correct as well. product_variant has missing values, but doesn't matter for now.

In [29]:
reviews_df.head(5)

,review_text,review_date,review_id,product_name,product_category,product_variant,product_price,product_url,product_id,rating,sold_count,shop_id,sentiment_label
0,baru sekali ini terima brg dr belanja online d...,2024-12-22,1134256160,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
1,cocok bgt aku sama telur nya. nga Amis menurut...,2025-02-25,1242584634,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
2,Telornya sudah sampai di rumah dengan kemasan ...,2025-07-15,1573444677,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
3,Telor sudah diterima dengan baik dan tidak ada...,2025-07-20,1581728541,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
4,"Alhamdulillah penjual amanah,Telor nya terbaik...",2023-04-24,881041355,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Full Design,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive


In [30]:
prod_sold_df = reviews_df[["product_id", "sold_count"]].copy()
prod_sold_df

,product_id,sold_count
0,4601033481,1000000
1,4601033481,1000000
2,4601033481,1000000
3,4601033481,1000000
4,4601033481,1000000
...,...,...
65538,15686645897,90
65539,15686645897,90
65540,15686645897,90
65541,15686645897,90


In [31]:
prod_sold_df = prod_sold_df.drop_duplicates(subset="product_id", keep="first")
prod_sold_df

,product_id,sold_count
0,4601033481,1000000
20,1457934746,100000
40,2349004012,100000
60,3648308521,100000
80,102061319829,50000
...,...,...
65494,102355223519,90
65495,1539738160,90
65498,102677804179,90
65518,16656205542,90


In [32]:
    sentiment_label_map = {
        "positive" : 1,
        "neutral" : 0,
        "negative" : -1
    }
    # getting product_id again, and the sum of their overall sentiment_label
    sentiment_score = reviews_df[["product_id", "sentiment_label"]].copy()
    sentiment_score

,product_id,sentiment_label
0,4601033481,positive
1,4601033481,positive
2,4601033481,positive
3,4601033481,positive
4,4601033481,positive
...,...,...
65538,15686645897,positive
65539,15686645897,positive
65540,15686645897,positive
65541,15686645897,positive


In [33]:
sentiment_score["sentiment_label"] = sentiment_score["sentiment_label"].map(sentiment_label_map).fillna(0) # fillna is for any uncertain and unknown value
sentiment_score

,product_id,sentiment_label
0,4601033481,1
1,4601033481,1
2,4601033481,1
3,4601033481,1
4,4601033481,1
...,...,...
65538,15686645897,1
65539,15686645897,1
65540,15686645897,1
65541,15686645897,1


In [34]:
sentiment_score = sentiment_score.groupby("product_id")["sentiment_label"].mean().reset_index() # sum the sentiment_score for each product_id, reset_index so that we can use product_id
sentiment_score

,product_id,sentiment_label
0,4298375,1.000000
1,12313359,1.000000
2,21172457,1.000000
3,22825312,0.888889
4,22845300,1.000000
...,...,...
5516,102656595877,1.000000
5517,102662280557,1.000000
5518,102671438736,1.000000
5519,102675049148,1.000000


In [35]:
prod_sold_score_df = pd.merge(prod_sold_df, sentiment_score, on="product_id", how="inner")
prod_sold_score_df

,product_id,sold_count,sentiment_label
0,4601033481,1000000,0.90
1,1457934746,100000,0.95
2,2349004012,100000,0.65
3,3648308521,100000,0.80
4,102061319829,50000,1.00
...,...,...,...
5516,102355223519,90,1.00
5517,1539738160,90,1.00
5518,102677804179,90,1.00
5519,16656205542,90,1.00


In [36]:
avg_vws = (prod_sold_score_df["sentiment_label"] * prod_sold_score_df["sold_count"]).sum() / prod_sold_score_df["sold_count"].sum()
avg_vws

np.float64(0.9427354616723245)